# Week 90, Day 1: Build

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w090d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('waymark_domain.py', 'w', encoding='utf-8') as f:
    f.write('import heapq\n\n\nclass UnknownPlace(LookupError):\n    pass\n\n\nclass NoRoute(Exception):\n    pass\n\n\ndef naismith_minutes(km, climb_m):\n    return round(km / 5 * 60 + climb_m / 600 * 60)\n\n\nclass Route:\n    def __init__(self, stops, km, climb_m):\n        self.stops, self.km, self.climb_m = list(stops), km, climb_m\n\n    @property\n    def minutes(self):\n        return naismith_minutes(self.km, self.climb_m)\n\n    def __repr__(self):\n        return f"Route({self.stops}, {self.km} km, {self.climb_m} m)"\n\n\nclass PathNetwork:\n    def __init__(self, heights):\n        self.heights = dict(heights)\n        self.paths = {name: {} for name in self.heights}\n\n    def _check(self, *names):\n        for name in names:\n            if name not in self.heights:\n                raise UnknownPlace(name)\n\n    def add_path(self, a, b, km):\n        self._check(a, b)\n        if km <= 0:\n            raise ValueError("km must be positive")\n        self.paths[a][b] = km\n        self.paths[b][a] = km\n\n    def climb(self, stops):\n        return sum(max(0, self.heights[b] - self.heights[a]) for a, b in zip(stops, stops[1:]))\n\n    def shortest(self, start, end):\n        self._check(start, end)\n        dist, prev, done = {start: 0}, {}, set()\n        heap = [(0, start)]\n        while heap:\n            d, node = heapq.heappop(heap)\n            if node in done:\n                continue\n            done.add(node)\n            if node == end:\n                break\n            for nxt, km in sorted(self.paths[node].items()):\n                if d + km < dist.get(nxt, float("inf")):\n                    dist[nxt] = d + km\n                    prev[nxt] = node\n                    heapq.heappush(heap, (d + km, nxt))\n        if end not in done:\n            raise NoRoute(f"no route from {start} to {end}")\n        stops = [end]\n        while stops[-1] != start:\n            stops.append(prev[stops[-1]])\n        stops.reverse()\n        return Route(stops, round(dist[end], 2), self.climb(stops))\n')
with open('waymark_map.py', 'w', encoding='utf-8') as f:
    f.write('from waymark_domain import PathNetwork\n\nHEIGHTS = {"Mill": 40, "Weir": 55, "Church": 80, "Ridge": 310, "Farm": 120, "Wood": 150, "Lake": 60, "Island": 5}\nPATHS = [("Mill", "Weir", 1.5), ("Weir", "Church", 2.0), ("Mill", "Church", 4.0), ("Church", "Ridge", 3.0),\n         ("Weir", "Farm", 2.5), ("Farm", "Ridge", 2.0), ("Farm", "Wood", 1.5), ("Wood", "Lake", 2.0), ("Lake", "Mill", 3.5)]\n\n\ndef sample_network():\n    network = PathNetwork(HEIGHTS)\n    for a, b, km in PATHS:\n        network.add_path(a, b, km)\n    return network\n')
with open('walks.py', 'w', encoding='utf-8') as f:
    f.write('WALKS = [\n    {"day": "2026-01-04", "km": 6.0, "climb_m": 270, "minutes": 105, "route": "Ridge walk"},\n    {"day": "2026-01-18", "km": 7.5, "climb_m": 40, "minutes": 98, "route": None},\n    {"day": "2026-02-01", "km": 11.0, "climb_m": 110, "minutes": 150, "route": "Lake loop"},\n    {"day": "2026-02-08", "km": 6.0, "climb_m": 270, "minutes": 101, "route": "Ridge walk"},\n    {"day": "2026-02-15", "km": 4.0, "climb_m": 0, "minutes": 50, "route": None},\n    {"day": "2026-03-29", "km": 15.5, "climb_m": 135, "minutes": 215, "route": None},\n    {"day": "2026-04-05", "km": 5.5, "climb_m": 110, "minutes": 80, "route": None},\n    {"day": "2026-04-12", "km": 13.0, "climb_m": 270, "minutes": 195, "route": None},\n    {"day": "2026-04-19", "km": 6.0, "climb_m": 270, "minutes": 97, "route": "Ridge walk"},\n]\n')

**Predict before you run:** what will this print?



In [ ]:
from itertools import accumulate
from waymark_map import sample_network

network = sample_network()
stops = ["Mill", "Weir", "Farm", "Ridge"]
steps = [network.paths[a][b] for a, b in zip(stops, stops[1:])]
print(list(accumulate(steps, initial=0)))
print([network.heights[s] for s in stops])

> **Key idea.** An elevation profile plots cumulative distance against height; label the axes with units, don't force the y axis to zero, and share scales when comparing. Load the log into pandas, parse dates, use `resample("MS")` for months (with gaps as zeros), `rolling("28D", on=...)` for recent totals and `assign` for derived columns; return plain Python from tested functions. Compare actual and estimated times with a scatter and the `actual = estimate` line. Separate preparing data from drawing it; test the first exactly and the second by inspecting the figure.

### Exercise: Profile data

Write `profile(network, stops)`: `(distances, heights)`, where `distances` is the cumulative distance at each stop (starting at 0, each rounded to 2 places) and `heights` the stops' heights. Then `steepest_section(network, stops)`: `(a, b, metres_per_km)` for the step with the largest climb per km (uphill only, rounded to a whole number; ties to the earlier step), or `None` if nothing climbs.

In [ ]:
from itertools import accumulate
from waymark_map import sample_network

def profile(network, stops):
    ___

def steepest_section(network, stops):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from waymark_map import sample_network\n\ndef test_profile():\n    \"distance along against height\"\n    net = sample_network()\n    assert profile(net, [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"]) == ([0, 1.5, 4.0, 6.0], [40, 55, 120, 310])\n    assert profile(net, [\"Mill\"]) == ([0], [40])\n    assert steepest_section(net, [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"]) == (\"Farm\", \"Ridge\", 95)\n    assert steepest_section(net, [\"Ridge\", \"Farm\", \"Weir\", \"Mill\"]) is None")

### Exercise: Drawing the profile

Write `draw_profile(distances, heights, stops, title)`: a figure with one axes, a line of `heights` against `distances` with circle markers, the area under it filled (`fill_between` down to the lowest height minus 20), each point annotated with its stop's name (`ax.annotate`), the title, x label `"distance (km)"` and y label `"height (m)"`, and the y axis from the lowest height minus 20 to the highest plus 40. Return `(fig, ax)`.

In [ ]:
import matplotlib.pyplot as plt

def draw_profile(distances, heights, stops, title):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_chart():\n    \"a labelled profile\"\n    fig, ax = draw_profile([0, 1.5, 4.0, 6.0], [40, 55, 120, 310], [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"], \"Mill to Ridge\")\n    line = ax.lines[0]\n    assert list(line.get_xdata()) == [0, 1.5, 4.0, 6.0] and list(line.get_ydata()) == [40, 55, 120, 310] and line.get_marker() == \"o\"\n    assert len(ax.collections) == 1, \"the filled area\"\n    assert [t.get_text() for t in ax.texts] == [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"]\n    assert ax.get_title() == \"Mill to Ridge\" and ax.get_xlabel() == \"distance (km)\" and ax.get_ylabel() == \"height (m)\"\n    assert ax.get_ylim() == (20.0, 350.0)")

### Exercise: The log in pandas

Write `log_frame(walks)`: a DataFrame of the walk dicts with `day` parsed as dates, sorted by day, plus columns `estimate` (Naismith minutes from the page's `waymark_domain`) and `ratio` (minutes / estimate, rounded to 2 places). Then `monthly_km(frame)`: a list of `(month, km)` for every month from the first walk's to the last's, with `month` as `"2026-01"` text and km rounded to 1 place (0.0 for empty months), using `resample("MS")`. And `rolling_km(frame, days=28)`: the rolling sum of km over the previous `days` days ending at each walk, as a list rounded to 1 place.

In [ ]:
import pandas as pd
from waymark_domain import naismith_minutes

def log_frame(walks):
    ___

def monthly_km(frame):
    ___

def rolling_km(frame, days=28):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from walks import WALKS\n\ndef test_frame():\n    \"dates, estimates, ratios\"\n    f = log_frame(list(reversed(WALKS)))\n    assert f[\"day\"].dt.strftime(\"%Y-%m-%d\").tolist()[:2] == [\"2026-01-04\", \"2026-01-18\"]\n    assert f[\"estimate\"].tolist()[:3] == [99, 94, 143] and f[\"ratio\"].tolist()[:3] == [1.06, 1.04, 1.05]\n\ndef test_summaries():\n    \"months with gaps, and the last four weeks\"\n    f = log_frame(WALKS)\n    assert monthly_km(f) == [(\"2026-01\", 13.5), (\"2026-02\", 21.0), (\"2026-03\", 15.5), (\"2026-04\", 24.5)]\n    assert rolling_km(f) == [6.0, 13.5, 18.5, 24.5, 21.0, 15.5, 21.0, 34.0, 40.0]\n    gap = log_frame([WALKS[0], {\"day\": \"2026-03-01\", \"km\": 3.0, \"climb_m\": 0, \"minutes\": 40, \"route\": None}])\n    assert monthly_km(gap) == [(\"2026-01\", 6.0), (\"2026-02\", 0.0), (\"2026-03\", 3.0)]")

### Exercise: Pace against the rule

Write `draw_pace(frame)`: a figure with one axes: a scatter of `estimate` (x) against `minutes` (y), the line `y = x` from the smallest to the largest of all those values, dashed and labelled `"Naismith's rule"`, the scatter labelled `"walks"`, a legend, title `"Actual against estimated time"`, and both axes labelled `"estimated minutes"` and `"actual minutes"`. Return `(fig, ax)`.

In [ ]:
import matplotlib.pyplot as plt

def draw_pace(frame):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_pace():\n    \"points and the rule\"\n    frame = pd.DataFrame({\"estimate\": [99, 94, 141], \"minutes\": [105, 98, 150]})\n    fig, ax = draw_pace(frame)\n    assert ax.collections[0].get_offsets().tolist() == [[99, 105], [94, 98], [141, 150]]\n    line = ax.lines[0]\n    assert list(line.get_xdata()) == [94, 150] and list(line.get_ydata()) == [94, 150] and line.get_linestyle() == \"--\"\n    assert [t.get_text() for t in ax.get_legend().get_texts()] == [\"walks\", \"Naismith's rule\"]\n    assert ax.get_xlabel() == \"estimated minutes\" and ax.get_ylabel() == \"actual minutes\"")

### Bug hunt: A hill flattened

Starting the profile's y axis at zero squashes a 270 m climb into a gentle slope (and wastes most of the chart): it prints `y from 0.0`. Start it just below the lowest height, so it prints `y from 20.0`.

In [ ]:
import matplotlib.pyplot as plt

heights = [40, 55, 120, 310]
fig, ax = plt.subplots()
ax.plot([0, 1.5, 4.0, 6.0], heights)
ax.set_ylim(0, max(heights) + 40)
print("y from", ax.get_ylim()[0])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"profiles zoom in, labelled\"\n    expect_output(\"y from 20.0\")")

### Bug hunt: February disappeared

Grouping by a month string only gives months that have walks, so the chart's February gap vanishes: it prints `['2026-01', '2026-03']`. Resample by month start so February appears with 0, and it prints `['2026-01', '2026-02', '2026-03']`.

In [ ]:
import pandas as pd

frame = pd.DataFrame({"day": pd.to_datetime(["2026-01-04", "2026-03-01"]), "km": [6.0, 3.0]})
months = frame.groupby(frame["day"].dt.strftime("%Y-%m"))["km"].sum()
print(list(months.index))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"every month\"\n    expect_output(\"['2026-01', '2026-02', '2026-03']\")")

### Puzzle: Prepare, then draw

Arrange the service for a route's profile: prepare the data in a tested function, then draw it, then return the figure. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
from itertools import accumulate
p = ___   # list(accumulate([1.5, 2.5, 2.0], initial=0))
q = ___   # True or False: an elevation profile's y axis must start at 0
r = ___   # the ratio, to 2 places, for a walk taking 105 minutes against an estimate of 99
s = ___   # months returned by resample("MS") for walks on 2026-01-04 and 2026-04-19
t = ___   # "above" or "below": where points for walks slower than the rule fall relative to y = x

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [0, 1.5, 4.0, 6.0] and q is False and r == 1.06, \"p, q and r\"\n    assert s == 4 and t == \"above\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the statistics page

Write `stats_page(walks, network, route_stops)` using the given helpers. Return `(summary, fig)`. `summary` has `walks`, `km` (1 place), `median_ratio` (2 places), `best_month` (the month text with most km, ties to the earlier) and `steepest` (from `steepest_section` on the route, as `"A to B (N m per km)"` or `"no climbs"`). `fig` is a 2 by 2 figure, titled `"Waymark statistics"` (with `suptitle`), with: top left, the route's profile (line with markers, x label `"distance (km)"`, y label `"height (m)"`, title `"Profile"`); top right, bars of km per month (x tick labels are the month texts, title `"Distance by month"`, y from 0); bottom left, the pace scatter with the `y = x` line (title `"Pace"`); bottom right, the rolling 28-day km as a line against the walks' dates (title `"Last 28 days"`).

In [ ]:
from itertools import accumulate
import matplotlib.pyplot as plt
import pandas as pd
from waymark_domain import naismith_minutes

def profile(network, stops):
    steps = [network.paths[a][b] for a, b in zip(stops, stops[1:])]
    return [round(d, 2) for d in accumulate(steps, initial=0)], [network.heights[s] for s in stops]

def steepest_section(network, stops):
    best = None
    for a, b in zip(stops, stops[1:]):
        rate = max(0, network.heights[b] - network.heights[a]) / network.paths[a][b]
        if rate > 0 and (best is None or rate > best[2]):
            best = (a, b, rate)
    return None if best is None else (best[0], best[1], round(best[2]))

def log_frame(walks):
    frame = pd.DataFrame(walks)
    frame["day"] = pd.to_datetime(frame["day"])
    frame = frame.sort_values("day").reset_index(drop=True)
    estimate = [naismith_minutes(k, c) for k, c in zip(frame["km"], frame["climb_m"])]
    return frame.assign(estimate=estimate, ratio=(frame["minutes"] / pd.Series(estimate)).round(2))

def monthly_km(frame):
    months = frame.set_index("day").resample("MS")["km"].sum()
    return [(d.strftime("%Y-%m"), round(float(v), 1)) for d, v in months.items()]

def rolling_km(frame, days=28):
    return [round(float(v), 1) for v in frame.rolling(f"{days}D", on="day")["km"].sum()]

def stats_page(walks, network, route_stops):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from walks import WALKS\nfrom waymark_map import sample_network\n\ndef test_summary():\n    \"the numbers\"\n    summary, fig = stats_page(WALKS, sample_network(), [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"])\n    assert summary == {\"walks\": 9, \"km\": 74.5, \"median_ratio\": 1.04, \"best_month\": \"2026-04\", \"steepest\": \"Farm to Ridge (95 m per km)\"}\n    flat, _ = stats_page(WALKS[:1], sample_network(), [\"Ridge\", \"Farm\"])\n    assert flat[\"steepest\"] == \"no climbs\" and flat[\"best_month\"] == \"2026-01\"\n\ndef test_figure():\n    \"four charts\"\n    summary, fig = stats_page(WALKS, sample_network(), [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"])\n    profile_ax, months_ax, pace_ax, recent_ax = fig.axes\n    assert [a.get_title() for a in fig.axes] == [\"Profile\", \"Distance by month\", \"Pace\", \"Last 28 days\"]\n    assert list(profile_ax.lines[0].get_ydata()) == [40, 55, 120, 310] and profile_ax.get_ylabel() == \"height (m)\"\n    assert [round(p.get_height(), 1) for p in months_ax.patches] == [13.5, 21.0, 15.5, 24.5] and months_ax.get_ylim()[0] == 0\n    fig.canvas.draw()\n    assert [t.get_text() for t in months_ax.get_xticklabels()] == [\"2026-01\", \"2026-02\", \"2026-03\", \"2026-04\"]\n    assert pace_ax.collections[0].get_offsets().shape == (9, 2) and len(pace_ax.lines) == 1\n    assert [round(v, 1) for v in recent_ax.lines[0].get_ydata()] == [6.0, 13.5, 18.5, 24.5, 21.0, 15.5, 21.0, 34.0, 40.0]\n    assert fig._suptitle.get_text() == \"Waymark statistics\"")

## Warm-up

Two lists, distance along the route and height at each stop, are an **elevation profile**. Drawn as a line, the steep last section to Ridge is obvious at a glance in a way that "270 m of climbing" isn't.

## An elevation profile

The profile's x values are the **cumulative distance** at each stop (`itertools.accumulate` with an initial 0), and the y values the heights. Draw it as a line with a lightly filled area underneath, label both axes with units, and mark the stops. Two honesty rules from Week 83 apply with a twist: a profile's y axis normally **doesn't** start at zero (that would flatten every hill), but it **must** be labelled so nobody mistakes a 40 m rise for a mountain; and two profiles shown side by side must share the same scales, or the flatter route can look steeper.

## The log as a DataFrame

The walk log (Week 88) becomes a DataFrame with one row per walk. Parse the `day` column into dates (`pd.to_datetime`), then:

- **by month**: `resample("MS")` on the dates gives one row per month, including months with no walks (as zeros), which `groupby` on a month string wouldn't;
- **rolling**: a 28-day rolling sum of distance (`rolling("28D", on="day")`) shows whether the walker is doing more or less lately;
- **derived columns** with `assign`: the Naismith estimate for each walk, and the ratio of actual to estimated time.

Return plain Python values (`tolist`, `to_dict`) from these functions, so the tests don't depend on pandas' printed formats, which differ between versions.

## Pace against the rule

A scatter plot of estimated against actual minutes, with the line `actual = estimate` drawn across it, shows a walker's pace at a glance: points above the line are walks slower than the rule. Day 2 of Week 88 summarised this as one number (the median ratio); the chart shows whether it's consistent or whether hilly walks are where the time goes.

## Prepare, then draw

Every chart in Waymark has two functions: one that **prepares the data** (pure, returns lists or a small DataFrame, easy to test exactly) and one that **draws it** (takes the prepared data, returns the figure, tested by inspecting the figure's lines, bars and labels). Mixing them makes both harder to test and to change. The dashboard in the challenge is then just a layout calling the drawing functions.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **One chart your users need**, with its data prepared by a tested function and drawn by another, inspected in a test.
2. **Honesty check** each chart against Week 83's rules (and today's exception for profiles).
3. **Merge** and update the burndown.

## Before you go

Tomorrow is S12: sharing a route with someone else, safely, with links that can expire and be revoked.